In [2]:
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 18.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import skfuzzy as fuzz

from skfuzzy import control as ctrl

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [8]:
iris = sns.load_dataset("iris")

dlugosc_platka = iris["petal_length"]
szerokosc_platka = iris["petal_width"]

x = np.column_stack((dlugosc_platka, szerokosc_platka))

mapowanie = {
    "setosa" : 0,
    "versicolor": 1,
    "virginica" : 2
}

y = iris["species"].map(mapowanie)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state = 42,
    stratify=y
)


petal_length = ctrl.Antecedent(np.linspace(0.5, 7.5, 1000), "Dlugosc platka")
petal_width = ctrl.Antecedent(np.linspace(0, 3, 1000), "Szerokosc platka")

species = ctrl.Consequent(np.linspace(0,2,1000), "Gatunek")



# Funkcje przynależności dla długości płatka # 0.5 - 7.5
petal_length["Mala"] = fuzz.trimf(petal_length.universe, [0.5, 0.5, 2.5])
petal_length["Srednia"] = fuzz.trimf(petal_length.universe, [2.3, 4.3, 5.5])
petal_length["Duza"] = fuzz.trimf(petal_length.universe, [4.7, 7.5, 7.5])

# Funkcje przynależności dla szerokości płatka
petal_width["Mala"] = fuzz.trimf(petal_width.universe, [0, 0, 0.9])
petal_width["Srednia"] = fuzz.trimf(petal_width.universe, [0.7, 1.3, 2])
petal_width["Duza"] = fuzz.trapmf(petal_width.universe, [1.4, 1.7, 2.3, 3])

# Funkcje przynależności dla gatunków
species["Setosa"] = fuzz.trimf(species.universe, [0, 0, 0.5])
species["Versicolor"] = fuzz.trimf(species.universe, [0.5, 1, 1.5])
species["Virginica"] = fuzz.trimf(species.universe, [1.5, 2, 2])


# Reguła klasyfikująca kwiat jako Setosa
rule1 = ctrl.Rule(
    petal_length["Mala"] | petal_width["Mala"],
    species["Setosa"]
)

# Reguła klasyfikująca kwiat jako Versicolor
rule2 = ctrl.Rule(
    petal_length["Srednia"] & petal_width["Srednia"],
    species["Versicolor"]
)

# Reguła klasyfikująca kwiat jako Virginica
rule3 = ctrl.Rule(
    petal_length["Duza"] | petal_width["Duza"],
    species["Virginica"]
)

In [9]:

# Utworzenie systemu rozmytego
species_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])

# Utworzenie symulacji systemu rozmytego
species_sim = ctrl.ControlSystemSimulation(species_ctrl)


# Lista na przewidywane klasy
y_pred = []

# Sprawdzenie systemu dla każdej próbki ze zbioru testowego
for length, width in x_test:

    # Przekazanie długości płatka do systemu
    species_sim.input["Dlugosc platka"] = length

    # Przekazanie szerokości płatka do systemu
    species_sim.input["Szerokosc platka"] = width

    # Wykonanie obliczeń systemu rozmytego
    species_sim.compute()

    # Pobranie wyniku systemu
    result = species_sim.output["Gatunek"]

    # Przypisanie wyniku do klasy Setosa
    if result < 0.5:
        y_pred.append(0)

    # Przypisanie wyniku do klasy Versicolor
    elif result < 1.5:
        y_pred.append(1)

    # Przypisanie wyniku do klasy Virginica
    else:
        y_pred.append(2)


# Wyświetlenie dokładności systemu
print("Accuracy:", accuracy_score(y_test, y_pred))

# Wyświetlenie metryk dla poszczególnych gatunków
print(classification_report(
    y_test,
    y_pred,
    target_names=["Setosa", "Versicolor", "Virginica"]
))

Accuracy: 0.9333333333333333
              precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        10
  Versicolor       0.83      1.00      0.91        10
   Virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

